# 11 — Physical Unclonable Functions and Hardware Fingerprinting

## What This Is
Physical Unclonable Functions (PUFs) exploit manufacturing process variations to create a unique hardware fingerprint — a digital "fingerprint" that cannot be cloned even by the manufacturer. Ring Oscillator PUFs, SRAM PUFs, and Arbiter PUFs are the main architectures.

## Why It Matters
PUFs enable authentication without storing secret keys (the secret IS the hardware). Applications: secure key generation, IC authentication, counterfeit detection, and device attestation. Smart cards and TPM 2.0 use SRAM PUF technology.

## Where the Community Stands
VERASEC (SRAM PUF) is deployed in STMicroelectronics and NXP chips. CHES proceedings track PUF research heavily. Machine learning-based PUF modelling attacks (MLPAs) can clone some PUF types — especially Arbiter PUFs — from challenge-response pairs, driving research into stronger PUF designs.

## Key Properties
- **Uniqueness**: Different devices produce different responses to the same challenge
- **Reliability**: Same device produces same response to same challenge (with noise)
- **Unpredictability**: Cannot predict response without physical access
- **Unclonability**: Manufacturing variations cannot be reproduced precisely

In [ ]:
import random
import math
import statistics
from dataclasses import dataclass, field
from typing import List, Dict, Tuple

# --- Ring Oscillator PUF simulation ---
# Real RO PUF: two ring oscillators per cell; compare frequencies
# Manufacturing variations cause one to be slightly faster — this is the bit value

class ROPUFChip:
    """Simulated Ring Oscillator PUF."""

    def __init__(self, chip_id: int, n_cells: int = 128,
                 noise_std: float = 0.02):
        self.chip_id   = chip_id
        self.n_cells   = n_cells
        self.noise_std = noise_std
        rng = random.Random(chip_id)  # unique manufacturing variations
        # Each cell: (freq_0, freq_1) — manufacturing process determines which is faster
        self.cells = [
            (rng.gauss(1.0, 0.05), rng.gauss(1.0, 0.05))
            for _ in range(n_cells)
        ]

    def challenge_response(self, challenge: List[int],
                            rng: random.Random = None) -> List[int]:
        """challenge[i] selects which RO pair to compare."""
        if rng is None:
            rng = random.Random(0)
        response = []
        for c in challenge[:self.n_cells]:
            cell_idx = c % self.n_cells
            f0, f1   = self.cells[cell_idx]
            # Add measurement noise
            noise    = rng.gauss(0, self.noise_std)
            bit      = 1 if (f0 - f1 + noise) > 0 else 0
            response.append(bit)
        return response

# Create 5 chips with same design, different manufacturing
chips = [ROPUFChip(chip_id=i, n_cells=64) for i in range(5)]
rng   = random.Random(99)
challenge = [rng.randint(0, 63) for _ in range(64)]

responses = [chip.challenge_response(challenge, rng) for chip in chips]

print(f'RO PUF responses to same challenge ({len(challenge)}-bit):')
for i, resp in enumerate(responses):
    bits = ''.join(str(b) for b in resp[:32])
    hw   = sum(resp)
    print(f'  Chip {i}: {bits}... (HW={hw}/64)')

In [ ]:
def fractional_hamming_distance(r1: List[int], r2: List[int]) -> float:
    """Proportion of bits that differ."""
    return sum(a != b for a, b in zip(r1, r2)) / len(r1)

# Intra-HD: same chip, different measurements (noise) — should be ~0
intra_hds = []
chip = chips[0]
for _ in range(20):
    r1 = chip.challenge_response(challenge, random.Random(random.randint(0,9999)))
    r2 = chip.challenge_response(challenge, random.Random(random.randint(0,9999)))
    intra_hds.append(fractional_hamming_distance(r1, r2))

# Inter-HD: different chips, same challenge — should be ~0.5
inter_hds = []
for i in range(len(chips)):
    for j in range(i+1, len(chips)):
        inter_hds.append(fractional_hamming_distance(responses[i], responses[j]))

print(f'PUF Quality Metrics:')
print(f'  Intra-HD (noise):   mean={statistics.mean(intra_hds):.4f}  std={statistics.stdev(intra_hds):.4f}')
print(f'  Inter-HD (unique):  mean={statistics.mean(inter_hds):.4f}  std={statistics.stdev(inter_hds):.4f}')
print(f'  Ideal intra-HD: ~0.000  (same chip, same response)')
print(f'  Ideal inter-HD: ~0.500  (different chips, unrelated responses)')

## PUF-Based Authentication Protocol
Authentication: verifier stores (challenge, response) pairs during enrollment. At authentication time, verifier sends challenge, device computes response, verifier checks with fuzzy comparison (allowing for noise).

In [ ]:
@dataclass
class PUFEnrollment:
    chip_id:    int
    crps:       List[Tuple[List[int], List[int]]]  # (challenge, response) pairs
    bit_errors: float  # measured intra-HD from enrollment

class PUFAuthenticator:
    def __init__(self, fuzzy_threshold: float = 0.15):
        self.threshold = fuzzy_threshold
        self.enrolled: Dict[int, PUFEnrollment] = {}

    def enroll(self, chip: ROPUFChip, n_crps: int = 10) -> PUFEnrollment:
        rng = random.Random(42)
        crps = []
        intra_hds = []
        for _ in range(n_crps):
            chal  = [rng.randint(0, chip.n_cells-1) for _ in range(chip.n_cells)]
            resp1 = chip.challenge_response(chal, random.Random(rng.randint(0,9999)))
            resp2 = chip.challenge_response(chal, random.Random(rng.randint(0,9999)))
            intra_hds.append(fractional_hamming_distance(resp1, resp2))
            crps.append((chal, resp1))
        enroll = PUFEnrollment(chip.chip_id, crps, statistics.mean(intra_hds))
        self.enrolled[chip.chip_id] = enroll
        return enroll

    def authenticate(self, chip: ROPUFChip) -> Dict:
        if chip.chip_id not in self.enrolled:
            return {'authenticated': False, 'reason': 'Not enrolled'}
        enrollment = self.enrolled[chip.chip_id]
        # Pick a random CRP from enrollment database
        chal, enrolled_resp = random.choice(enrollment.crps)
        # Device computes fresh response
        fresh_resp = chip.challenge_response(chal, random.Random(random.randint(0,99999)))
        hd = fractional_hamming_distance(enrolled_resp, fresh_resp)
        authenticated = hd <= self.threshold
        return {'authenticated': authenticated, 'hamming_distance': round(hd, 4),
                'threshold': self.threshold, 'chip_id': chip.chip_id}

auth = PUFAuthenticator(fuzzy_threshold=0.15)
# Enroll chips 0 and 1
for i in [0, 1]:
    auth.enroll(chips[i])

print('PUF Authentication Results:')
# Legitimate chip
r = auth.authenticate(chips[0])
print(f'  Enrolled chip 0:      auth={r["authenticated"]}  HD={r["hamming_distance"]}')
# Another enrolled chip
r = auth.authenticate(chips[1])
print(f'  Enrolled chip 1:      auth={r["authenticated"]}  HD={r["hamming_distance"]}')
# Unenrolled chip (different manufacturing)
r = auth.authenticate(chips[4])
print(f'  Unenrolled chip 4:    auth={r["authenticated"]}  reason={r.get("reason","")}')

## ML-Based PUF Modelling Attack
Arbiter PUFs can be cloned using machine learning: collect challenge-response pairs, train a logistic regression model, and predict responses without physical access. This is why XOR PUFs and PUF-as-challenge designs were developed.

In [ ]:
# Simplified Arbiter PUF model
class ArbiterPUF:
    """Simplified 1-bit arbiter PUF. Real design uses delay chain XOR."""

    def __init__(self, n_stages: int, chip_id: int):
        self.n_stages = n_stages
        rng = random.Random(chip_id)
        # Delay parameters for each stage (manufacturing variation)
        self.delays = [(rng.gauss(0, 1), rng.gauss(0, 1)) for _ in range(n_stages)]

    def respond(self, challenge: List[int], noise: float = 0.1,
                rng: random.Random = None) -> int:
        if rng is None:
            rng = random.Random(0)
        # Compute delay difference along chain
        delta = 0.0
        for i, c in enumerate(challenge[:self.n_stages]):
            d0, d1 = self.delays[i]
            delta += d1 - d0 if c == 1 else d0 - d1
        return 1 if delta + rng.gauss(0, noise) > 0 else 0

# Collect CRPs and attempt ML modelling
PUF_STAGES = 16
apuf  = ArbiterPUF(PUF_STAGES, chip_id=7)
rng   = random.Random(42)

# Generate training data
N_TRAIN = 500
X_train = [[rng.randint(0,1) for _ in range(PUF_STAGES)] for _ in range(N_TRAIN)]
y_train = [apuf.respond(x, rng=rng) for x in X_train]

# Simple logistic regression (gradient descent)
import math
w = [0.0] * PUF_STAGES
b = 0.0
lr = 0.05

def sigmoid(z): return 1/(1+math.exp(-max(-500, min(500, z))))

for epoch in range(200):
    dw = [0.0]*PUF_STAGES
    db = 0.0
    for x, y in zip(X_train, y_train):
        z = sum(wi*xi for wi, xi in zip(w, x)) + b
        pred = sigmoid(z)
        err  = pred - y
        for i in range(PUF_STAGES):
            dw[i] += err * x[i]
        db += err
    n = len(X_train)
    w  = [wi - lr*dw[i]/n for i, wi in enumerate(w)]
    b -= lr * db/n

# Test accuracy on new CRPs
N_TEST = 200
X_test  = [[rng.randint(0,1) for _ in range(PUF_STAGES)] for _ in range(N_TEST)]
y_test  = [apuf.respond(x, rng=rng) for x in X_test]
correct = sum(
    (1 if sum(wi*xi for wi,xi in zip(w,x))+b > 0 else 0) == y
    for x, y in zip(X_test, y_test)
)
acc = correct / N_TEST
print(f'Arbiter PUF ML Modelling Attack:')
print(f'  Training CRPs:  {N_TRAIN}')
print(f'  Test accuracy:  {acc:.3f} (0.5 = random guess, 1.0 = perfect clone)')
print(f'  Attack effective: {acc > 0.65}')
print(f'  Defence: XOR PUFs, feed-forward PUFs, protocol-level CRP hiding')